# Estimating the pairwise Granger coefficents across International Markets for the Vietnam Global Contagion Indicator (VGCI).

In [ ]:
from google.colab import files
from google.colab import drive
import pandas as pd
import os
import io
import numpy as np
import statsmodels.api as sm

In [ ]:
def import_data(file_path):
  try:
    drive.mount('/content/drive', force_remount=True)
    # Check if file exists
    if os.path.exists(file_path):
      df = pd.read_parquet(file_path)
      print(f"Loaded dataframe from Drive ({file_path})")
    else:
      raise FileNotFoundError(f"File not found at {file_path}")

  except Exception as e:
    print(f"Drive not available or file missing: {e}")
    print("Please upload dataframe manually.")
    uploaded = files.upload()

    # Automatically read the uploaded file
    file_name = list(uploaded.keys())[0]  # pick the first uploaded file
    try:
      df = pd.read_parquet(io.BytesIO(uploaded[file_name]))
    except:
      print("Wrong file extension. Parquet file required.")
    print(f"Loaded {file_name} from manual upload.")
    return df

In [ ]:
# Import the Dataframe with the international market indexes
file_path = "..."
df_glob = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_glob.parquet to df_glob.parquet
Loaded df_glob.parquet from manual upload.


In [ ]:
# Data cleaning and transformation
df_glob.ffill(inplace=True)
df_glob_ret = df_glob.copy()
df_glob_ret = df_glob_ret.dropna()
df_glob_ret = df_glob_ret.set_index('Date')
df_glob_ret = np.log(df_glob_ret / df_glob_ret.shift(1))
df_glob_ret = df_glob_ret.dropna()
df_glob_ret.columns = df_glob_ret.columns.str.replace('_Close', '', regex=False)
df_glob_ret.columns = df_glob_ret.columns.str.replace('_close', '', regex=False)
df_glob_ret.columns = df_glob_ret.columns.str.replace('^', '', regex=False)
df_glob_ret

,VNINDEX,AXJO,FTSE,GSPC,GSPTSE,HSI,KS11,N225,STOXX50E,TWII,CSI300
Date,,,,,,,,,,,
2012-01-05,-0.022907,-0.010828,-0.007828,0.002939,0.000891,0.004587,-0.001330,-0.008376,-0.014635,0.006738,-0.009778
2012-01-06,-0.012425,-0.008290,0.004506,-0.002540,-0.003996,-0.011781,-0.011115,-0.011655,-0.007412,-0.001453,0.006226
2012-01-09,0.007662,-0.000755,-0.006642,0.002259,0.000664,0.014558,-0.009075,0.000000,-0.005322,-0.003865,0.033472
2012-01-10,0.015673,0.011335,0.014927,0.008847,0.006049,0.007318,0.014529,0.003796,0.026338,0.012028,0.032719
2012-01-11,0.007947,0.008466,-0.004557,0.000310,-0.000799,0.007740,-0.004147,0.003037,-0.003397,0.001300,-0.004809
...,...,...,...,...,...,...,...,...,...,...,...
2026-03-25,0.026534,0.021043,0.014109,0.005404,0.013712,0.010803,0.015772,0.028253,0.012117,0.025038,0.013924
2026-03-26,-0.008211,-0.003735,-0.013407,-0.017559,-0.015407,-0.019108,-0.032743,-0.002719,-0.014873,-0.003040,-0.013296
2026-03-27,0.016983,-0.001103,-0.000481,-0.016863,0.002293,0.003833,-0.003962,-0.004311,-0.010862,-0.006773,0.005576


Estimate the pairwise Granger coefficents across international markets on 250-day rolling windows, estimating the best lag with Akaike information criterion (AIC).

In [ ]:
window = 250 # length of rolling windows
max_lag = 5 # 1 week of trading activities
out = []

# 250-day rolling windows
for t in range(window, len(df_glob_ret)):
  w = df_glob_ret.iloc[t-window:t]
  date = df_glob_ret.index[t]
  # precompute once per window (speed-up)
  w_lagged = {i: w.shift(i) for i in range(1, max_lag+1)}

  # pairwise loops:
  for y in df_glob_ret.columns: # y = caused
    for x in df_glob_ret.columns: # x = causing
      if x == y:
        continue

      try:
        best_aic = float('inf')
        best_res = None
        best_p = 1
        best_sig_aic = float('inf')
        best_sig_res = None
        best_sig_p = None

        # lag selection loop
        for p in range(1, max_lag+1):
          Y = w[y].iloc[p:]
          Y_lags = [w_lagged[i][y] for i in range(1, p+1)]
          X_lags = [w_lagged[i][x] for i in range(1, p+1)]
          X = pd.concat(Y_lags + X_lags, axis=1).iloc[p:]
          X = sm.add_constant(X)
          # store positions explicitly
          start_x = 1 + len(Y_lags)
          end_x = start_x + len(X_lags)

          res = sm.OLS(Y.values, X.values).fit()

          # get best AIC
          if res.aic < best_aic:
            best_aic = res.aic
            best_res = res
            best_p = p
            best_start_x = start_x
            best_end_x = end_x

          # Joint test on X lags
          k = len(res.params)
          R = np.zeros((p, k))
          R[:, start_x:end_x] = np.eye(p)
          p_val = float(res.f_test(R).pvalue)

          # best AIC among significant ones
          if p_val < 0.05:
            if res.aic < best_sig_aic:
              best_sig_aic = res.aic
              best_sig_res = res
              best_sig_p = p
              best_sig_start_x = start_x
              best_sig_end_x = end_x

        # choose significant model if exists
        if best_sig_res is not None:
          best_res = best_sig_res
          best_p = best_sig_p
          best_start_x = best_sig_start_x
          best_end_x = best_sig_end_x

        # recompute p-value for the chosen model
        k = len(best_res.params)
        R = np.zeros((best_p, k))
        R[:, best_start_x: best_end_x] =  np.eye(best_p)
        p_value = float(best_res.f_test(R).pvalue)

        # store (kept your original coef choice)
        out.append({'date': date,
                    'causing': x,
                    'caused': y,
                    'lag': best_p,
                    'coef': best_res.params[best_start_x:best_end_x].sum(),
                    'p_value': p_value})

      except Exception as e:
        print(e)
        break

gc_rolling_df = pd.DataFrame(out)

In [ ]:
gc_rolling_df.to_csv("gc_rolling_df.csv", index=False)
files.download("gc_rolling_df.csv")
gc_rolling_df.to_parquet("gc_rolling_df.parquet", index=False)
files.download("gc_rolling_df.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>